Repartition, save as parquet, and check average file size

In [1]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
# Initialize Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/19 19:16:26 WARN Utils: Your hostname, JACK2000-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.158 instead (on interface en0)
26/06/19 19:16:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/19 19:16:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
df = spark.read.parquet("../data/raw/yellow/2025/yellow_tripdata_2025-11.parquet")
df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [7]:
(df.repartition(4)
   .write.mode("overwrite")
   .parquet("data/processed/yellow/2025/11")
   )

In [8]:
# Inspect output part files
!ls -lh data/processed/yellow/2025/11/part-*.parquet

# average size is `total size of part files / 4`

-rw-r--r--@ 1 jack2000  staff    24M Jun 19 19:17 data/processed/yellow/2025/11/part-00000-31c7f7e0-47d2-47e7-b41b-d50d72e2885b-c000.snappy.parquet
-rw-r--r--@ 1 jack2000  staff    24M Jun 19 19:17 data/processed/yellow/2025/11/part-00001-31c7f7e0-47d2-47e7-b41b-d50d72e2885b-c000.snappy.parquet
-rw-r--r--@ 1 jack2000  staff    24M Jun 19 19:17 data/processed/yellow/2025/11/part-00002-31c7f7e0-47d2-47e7-b41b-d50d72e2885b-c000.snappy.parquet
-rw-r--r--@ 1 jack2000  staff    24M Jun 19 19:17 data/processed/yellow/2025/11/part-00003-31c7f7e0-47d2-47e7-b41b-d50d72e2885b-c000.snappy.parquet


In [9]:
# Or we can use Python to calculate the average size of part files
import glob, os
parts = glob.glob("data/processed/yellow/2025/11/part-*.parquet")
avg_mb = sum(os.path.getsize(p) for p in parts) / len(parts) / 1024 / 1024
print(f"{len(parts)} files, avg {avg_mb:.1f} MB")

4 files, avg 24.4 MB
